<a href="https://colab.research.google.com/github/nandini-012/Data-AI-Challenge-AI-Recruiter-/blob/main/notebooks/ravi_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/india_runs/India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl"  # Corrected path based on 'find' command
JD_KEYWORDS = [
    "machine learning", "ml", "ai", "artificial intelligence", "deep learning",
    "nlp", "natural language processing", "llm", "large language model",
    "computer vision", "cv", "data science", "mlops", "ai engineer",
    "ml engineer", "research scientist", "applied scientist"
]

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Debugging `FileNotFoundError`

Since the `FileNotFoundError` persists, let's verify the contents of the directory where `candidates.jsonl` is expected to be. This will help you confirm the correct file name and path.

In [16]:
import os

# Extract the directory path from DATA_PATH
directory_path = os.path.dirname(DATA_PATH)

print(f"Listing contents of directory: {directory_path}")
try:
    # List all files and directories in the specified path
    for item in os.listdir(directory_path):
        print(item)
except FileNotFoundError:
    print(f"Error: Directory not found at {directory_path}. Please check the path.")
except Exception as e:
    print(f"An unexpected error occurred: {e}")

# After inspecting the output, please ensure the DATA_PATH variable in the cell `OEG8yBWF2WH3`
# is updated to the correct, full path of your `candidates.jsonl` file.

Listing contents of directory: /content/drive/MyDrive/india_runs/India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge
candidate_schema.json
candidates.jsonl
job_description.docx
submission_spec.docx
sample_candidates.json
sample_submission.csv
redrob_signals_doc.docx
submission_metadata_template.yaml
README.docx
validate_submission.py


In [17]:
import json
import re
import numpy as np
import pandas as pd
from datetime import datetime, date

pd.set_option("display.max_columns", 50)

In [18]:
candidates = []
with open(DATA_PATH, 'r') as f:
    for line in f:
        candidates.append(json.loads(line))

print(f"Loaded {len(candidates)} candidates from {DATA_PATH}")

Loaded 100000 candidates from /content/drive/MyDrive/india_runs/India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl


In [19]:
import subprocess
result = subprocess.run(['find', '/content/drive/MyDrive/india_runs', '-iname', 'candidates.jsonl'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

/content/drive/MyDrive/india_runs/India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl




In [24]:
def behavioral_multiplier(row):
    mult = 1.0
    # recency penalty
    if row["days_since_active"] > 180:
        mult *= 0.6
    elif row["days_since_active"] > 90:
        mult *= 0.85
    # responsiveness boost/penalty
    mult *= (0.7 + 0.5 * (row["recruiter_response_rate"] or 0))  # 0.7x to 1.2x
    # interview completion
    if row["interview_completion_rate"] < 0.5:
        mult *= 0.9
    # open to work boost
    if row["open_to_work_flag"]:
        mult *= 1.05
    return round(mult, 3)

df["behavioral_multiplier"] = df.apply(behavioral_multiplier, axis=1)

In [25]:
def detect_honeypot_flags(row):
    flags = []
    cid = row["candidate_id"]
    exp_months = (row["years_of_experience"] or 0) * 12

    # 1. Skill duration longer than total claimed experience (bigger buffer now)
    for s in skills_lookup.get(cid, []):
        dm = s.get("duration_months", 0) or 0
        if dm > exp_months + 24:  # was +6, now +24 — only flag clear impossibilities
            flags.append(f"skill '{s.get('name')}' duration ({dm}mo) far exceeds experience ({exp_months:.0f}mo)")

    for s in skills_lookup.get(cid, []):
        if s.get("proficiency") == "expert" and (s.get("duration_months", 0) or 0) <= 2:
            flags.append(f"'expert' in '{s.get('name')}' with only {s.get('duration_months')}mo experience")

    career_months_sum = sum(j.get("duration_months", 0) or 0 for j in career_lookup.get(cid, []))
    if abs(career_months_sum - exp_months) > 48:  # was 36, now 48 — only severe mismatches
        flags.append(f"career history sums to {career_months_sum}mo vs stated {exp_months:.0f}mo experience")

    current_jobs = [j for j in career_lookup.get(cid, []) if j.get("is_current")]
    if len(current_jobs) > 1:
        flags.append(f"{len(current_jobs)} simultaneous current jobs")

    for e in education_lookup.get(cid, []):
        if e.get("end_year") and e.get("start_year") and e["end_year"] < e["start_year"]:
            flags.append(f"education end_year {e['end_year']} before start_year {e['start_year']}")

    return flags

df["honeypot_flags"] = df.apply(detect_honeypot_flags, axis=1)
# Require at least 2 flags OR one severe flag, not just any single mild flag
df["is_honeypot"] = df["honeypot_flags"].apply(lambda x: len(x) >= 2)
print(f"Flagged honeypots: {df['is_honeypot'].sum()} / {len(df)} ({df['is_honeypot'].mean()*100:.2f}%)")

Flagged honeypots: 473 / 100000 (0.47%)


In [26]:
def flatten_candidate(c):
    profile = c.get("profile", {})
    signals = c.get("redrob_signals", {})
    career = c.get("career_history", [])
    skills = c.get("skills", [])
    education = c.get("education", [])

    total_career_months = sum(j.get("duration_months", 0) or 0 for j in career)
    num_jobs = len(career)
    current_jobs = [j for j in career if j.get("is_current")]
    num_current_jobs = len(current_jobs)

    return {
        "candidate_id": c.get("candidate_id"),
        "headline": profile.get("headline", ""),
        "summary": profile.get("summary", ""),
        "current_title": profile.get("current_title", ""),
        "current_company": profile.get("current_company", ""),
        "current_company_size": profile.get("current_company_size", ""),
        "current_industry": profile.get("current_industry", ""),
        "location": profile.get("location", ""),
        "country": profile.get("country", ""),
        "years_of_experience": profile.get("years_of_experience", 0),
        "num_jobs": num_jobs,
        "num_current_jobs": num_current_jobs,
        "total_career_months": total_career_months,
        "num_skills": len(skills),
        "num_education": len(education),
        "top_education_tier": min([e.get("tier", "unknown") for e in education], default="unknown"),
        "profile_completeness_score": signals.get("profile_completeness_score", 0),
        "last_active_date": signals.get("last_active_date"),
        "open_to_work_flag": signals.get("open_to_work_flag", False),
        "recruiter_response_rate": signals.get("recruiter_response_rate", 0),
        "avg_response_time_hours": signals.get("avg_response_time_hours", 0),
        "connection_count": signals.get("connection_count", 0),
        "notice_period_days": signals.get("notice_period_days", 0),
        "willing_to_relocate": signals.get("willing_to_relocate", False),
        "github_activity_score": signals.get("github_activity_score", -1),
        "search_appearance_30d": signals.get("search_appearance_30d", 0),
        "saved_by_recruiters_30d": signals.get("saved_by_recruiters_30d", 0),
        "interview_completion_rate": signals.get("interview_completion_rate", 0),
        "offer_acceptance_rate": signals.get("offer_acceptance_rate", -1),
        "verified_email": signals.get("verified_email", False),
        "verified_phone": signals.get("verified_phone", False),
    }

df = pd.DataFrame([flatten_candidate(c) for c in candidates])
print(f"DataFrame created with shape: {df.shape}")
df.head()

DataFrame created with shape: (100000, 31)


,candidate_id,headline,summary,current_title,current_company,current_company_size,current_industry,location,country,years_of_experience,num_jobs,num_current_jobs,total_career_months,num_skills,num_education,top_education_tier,profile_completeness_score,last_active_date,open_to_work_flag,recruiter_response_rate,avg_response_time_hours,connection_count,notice_period_days,willing_to_relocate,github_activity_score,search_appearance_30d,saved_by_recruiters_30d,interview_completion_rate,offer_acceptance_rate,verified_email,verified_phone
0,CAND_0000001,"Backend Engineer | SQL, Spark, Cloud",Software / data professional with 6.9 years of...,Backend Engineer,Mindtree,10001+,IT Services,Toronto,Canada,6.9,2,1,82,17,1,tier_3,86.9,2026-05-20,True,0.34,177.8,356,60,False,9.2,249,4,0.71,0.58,True,True
1,CAND_0000002,Operations Manager | 12.5+ yrs experience,Professional with 12.5+ years of experience. M...,Operations Manager,Wipro,10001+,IT Services,"Chennai, Tamil Nadu",India,12.5,4,1,149,9,1,tier_4,78.7,2025-11-12,True,0.29,171.6,179,60,False,-1.0,107,10,0.62,-1.00,False,False
2,CAND_0000003,Customer Support | 1.1+ yrs experience,Professional with 1.1+ years of experience. I'...,Customer Support,TCS,10001+,IT Services,Austin,USA,1.1,1,1,13,6,2,tier_3,31.9,2026-03-21,False,0.46,119.4,19,150,False,-1.0,28,4,0.86,-1.00,True,False
3,CAND_0000004,Marketing Manager | Driving business outcomes,Professional with 3.8+ years of experience. My...,Marketing Manager,Dunder Mifflin,201-500,Paper Products,Sydney,Australia,3.8,3,1,44,10,2,tier_3,28.5,2026-03-25,False,0.26,104.1,485,120,True,-1.0,5,8,0.35,-1.00,True,True
4,CAND_0000005,Accountant | Helping teams scale,Professional with 11.0+ years of experience. I...,Accountant,Stark Industries,1001-5000,Manufacturing,"Gurgaon, Haryana",India,11.0,4,1,130,6,1,tier_3,84.6,2025-10-01,True,0.37,116.7,300,30,True,-1.0,67,1,0.74,-1.00,False,True


In [27]:
# 1. Initialize Feature Engineering (Lookups and Recency)
print("Calculating recency and initializing lookups...")

# Create Lookups
skills_lookup = {c['candidate_id']: c.get('skills', []) for c in candidates}
career_lookup = {c['candidate_id']: c.get('career_history', []) for c in candidates}
education_lookup = {c['candidate_id']: c.get('education', []) for c in candidates}

# Calculate days_since_active
df['last_active_date'] = pd.to_datetime(df['last_active_date'])
today = pd.Timestamp(date.today())
df['days_since_active'] = (today - df['last_active_date']).dt.days

print(f"Median days since last active: {df['days_since_active'].median()}")

Calculating recency and initializing lookups...
Median days since last active: 131.0


In [28]:
# 2. Apply Scoring Logic and Honeypot Detection
print("Applying scoring models and detecting honeypots...")

# Define and apply title relevance
def title_relevance_score(title):
    title_l = (title or "").lower()
    strong = ["ai engineer", "ml engineer", "machine learning engineer", "research scientist", "applied scientist"]
    if any(k in title_l for k in strong): return 1.0
    if any(k in title_l for k in ["data scientist", "ai", "ml", "machine learning"]): return 0.7
    return 0.2

df['title_score'] = df['current_title'].apply(title_relevance_score)

# Define and apply Behavioral Multiplier
def behavioral_multiplier(row):
    mult = 1.0
    if row['days_since_active'] > 180: mult *= 0.6
    elif row['days_since_active'] > 90: mult *= 0.85
    mult *= (0.7 + 0.5 * (row['recruiter_response_rate'] or 0))
    if row['open_to_work_flag']: mult *= 1.05
    return round(mult, 3)

df['behavioral_multiplier'] = df.apply(behavioral_multiplier, axis=1)

# Apply Honeypot Logic
def simple_honeypot(row):
    cid = row['candidate_id']
    exp_months = (row['years_of_experience'] or 0) * 12
    flags = 0
    if any((s.get('duration_months', 0) or 0) > exp_months + 24 for s in skills_lookup.get(cid, [])): flags += 1
    return flags >= 1

df['is_honeypot'] = df.apply(simple_honeypot, axis=1)
print(f"Flagged {df['is_honeypot'].sum()} honeypots.")

Applying scoring models and detecting honeypots...
Flagged 2821 honeypots.


In [29]:
# 3. Final Ranking and Display
print("Generating Final Top 100 Ranking...")

df['final_score'] = df['title_score'] * df['behavioral_multiplier']
ranked = df[~df['is_honeypot']].sort_values('final_score', ascending=False).reset_index(drop=True)
top_100 = ranked.head(100).copy()
top_100['rank'] = range(1, 101)

# Display the top 10 results for visibility
display(top_100[['rank', 'candidate_id', 'current_title', 'final_score']].head(10))

Generating Final Top 100 Ranking...


,rank,candidate_id,current_title,final_score
0,1,CAND_0077337,Staff Machine Learning Engineer,1.234
1,2,CAND_0050553,ML Engineer,1.234
2,3,CAND_0033445,ML Engineer,1.228
3,4,CAND_0010603,ML Engineer,1.228
4,5,CAND_0064888,ML Engineer,1.218
5,6,CAND_0043637,Junior ML Engineer,1.218
6,7,CAND_0080472,Junior ML Engineer,1.213
7,8,CAND_0080534,ML Engineer,1.213
8,9,CAND_0012840,Junior ML Engineer,1.208
9,10,CAND_0023587,ML Engineer,1.208


In [30]:
def is_relevant_title(title):
    title_l = (title or "").lower()
    return any(kw in title_l for kw in ["ai", "ml", "machine learning", "data scientist", "research scientist", "applied scientist"])

df["title_looks_relevant"] = df["current_title"].apply(is_relevant_title)
print("Relevant-looking titles:", df["title_looks_relevant"].sum(), "/", len(df))
print(df["current_title"].value_counts().head(20))

Relevant-looking titles: 978 / 100000
current_title
Business Analyst        5833
HR Manager              5830
Mechanical Engineer     5791
Accountant              5764
Project Manager         5754
Customer Support        5750
Operations Manager      5744
Content Writer          5727
Sales Executive         5713
Civil Engineer          5702
Graphic Designer        5689
Marketing Manager       5524
Software Engineer       3450
Full Stack Developer    2873
Cloud Engineer          2836
Java Developer          2809
.NET Developer          2788
DevOps Engineer         2787
Mobile Developer        2757
Frontend Engineer       2738
Name: count, dtype: int64


In [31]:
print("Open to work:", df["open_to_work_flag"].mean())
print("Recruiter response rate (mean):", df["recruiter_response_rate"].mean())
df["last_active_date"] = pd.to_datetime(df["last_active_date"])
today = pd.Timestamp(date.today())
df["days_since_active"] = (today - df["last_active_date"]).dt.days
print("Median days since last active:", df["days_since_active"].median())
print("Inactive >180 days:", (df["days_since_active"] > 180).sum())

Open to work: 0.35339
Recruiter response rate (mean): 0.4365736
Median days since last active: 131.0
Inactive >180 days: 28182


In [32]:
skills_lookup = {c["candidate_id"]: c.get("skills", []) for c in candidates}
career_lookup = {c["candidate_id"]: c.get("career_history", []) for c in candidates}
education_lookup = {c["candidate_id"]: c.get("education", []) for c in candidates}

def detect_honeypot_flags(row):
    flags = []
    cid = row["candidate_id"]
    exp_months = (row["years_of_experience"] or 0) * 12

    # 1. Skill duration longer than total claimed experience
    for s in skills_lookup.get(cid, []):
        dm = s.get("duration_months", 0) or 0
        if dm > exp_months + 6:  # small buffer for rounding
            flags.append(f"skill '{s.get('name')}' duration ({dm}mo) exceeds total experience ({exp_months:.0f}mo)")

    # 2. "Expert" proficiency claimed with near-zero duration
    for s in skills_lookup.get(cid, []):
        if s.get("proficiency") == "expert" and (s.get("duration_months", 0) or 0) <= 2:
            flags.append(f"'expert' in '{s.get('name')}' with only {s.get('duration_months')}mo experience")

    # 3. Sum of career_history durations wildly mismatched vs years_of_experience
    career_months_sum = sum(j.get("duration_months", 0) or 0 for j in career_lookup.get(cid, []))
    if abs(career_months_sum - exp_months) > 36:  # >3 year mismatch
        flags.append(f"career history sums to {career_months_sum}mo vs stated {exp_months:.0f}mo experience")

    # 4. Multiple simultaneous "current" jobs
    current_jobs = [j for j in career_lookup.get(cid, []) if j.get("is_current")]
    if len(current_jobs) > 1:
        flags.append(f"{len(current_jobs)} simultaneous current jobs")

    # 5. Education end_year before start_year, or impossible years
    for e in education_lookup.get(cid, []):
        if e.get("end_year") and e.get("start_year") and e["end_year"] < e["start_year"]:
            flags.append(f"education end_year {e['end_year']} before start_year {e['start_year']}")

    return flags

df["honeypot_flags"] = df.apply(detect_honeypot_flags, axis=1)
df["is_honeypot"] = df["honeypot_flags"].apply(lambda x: len(x) > 0)

print(f"Flagged honeypots: {df['is_honeypot'].sum()} / {len(df)} ({df['is_honeypot'].mean()*100:.1f}%)")
print("\nExample flagged candidates:")
print(df[df["is_honeypot"]][["candidate_id", "honeypot_flags"]].head(10).to_string())

# Save honeypot blacklist — use this to EXCLUDE candidates before ranking
honeypot_ids = set(df[df["is_honeypot"]]["candidate_id"])
print(f"\nHoneypot ID set ready: {len(honeypot_ids)} candidates will be excluded from ranking.")

Flagged honeypots: 13511 / 100000 (13.5%)

Example flagged candidates:
    candidate_id                                                                                                                                                                                                                                                               honeypot_flags
2   CAND_0000003                                                                                                                                                                                                         [skill 'Kubernetes' duration (34mo) exceeds total experience (13mo)]
10  CAND_0000011                                                      [skill 'Recommendation Systems' duration (40mo) exceeds total experience (24mo), skill 'Spring Boot' duration (31mo) exceeds total experience (24mo), skill 'Kubeflow' duration (59mo) exceeds total experience (24mo)]
11  CAND_0000012      [skill 'AWS' duration (30mo) exceeds total experi

In [33]:
def title_relevance_score(title):
    title_l = (title or "").lower()
    strong = ["ai engineer", "ml engineer", "machine learning engineer", "research scientist", "applied scientist"]
    medium = ["data scientist", "ai", "ml", "machine learning"]
    if any(k in title_l for k in strong):
        return 1.0
    if any(k in title_l for k in medium):
        return 0.7
    return 0.2

df["title_score"] = df["current_title"].apply(title_relevance_score)

def career_substance_score(cid):
    """Looks for production/deployment language in career_history descriptions,
    not just buzzword presence. Counts as a proxy for hands-on depth."""
    production_terms = ["deployed", "production", "shipped", "scaled", "users",
                         "latency", "throughput", "pipeline", "infrastructure", "launched"]
    text = " ".join(j.get("description", "") for j in career_lookup.get(cid, [])).lower()
    hits = sum(1 for t in production_terms if t in text)
    return min(hits / 5, 1.0)  # cap at 1.0

df["career_substance_score"] = df["candidate_id"].apply(career_substance_score)

def skill_match_score(cid):
    """Weighted by duration_months + endorsements, not just presence —
    this is what kills keyword-stuffer profiles."""
    relevant = ["python", "pytorch", "tensorflow", "machine learning", "deep learning",
                "nlp", "llm", "transformers", "computer vision", "mlops", "sql", "spark"]
    skills = skills_lookup.get(cid, [])
    score = 0
    for s in skills:
        name_l = (s.get("name") or "").lower()
        if any(r in name_l for r in relevant):
            weight = min((s.get("duration_months", 0) or 0) / 24, 1.0)  # 2yrs = full weight
            endorsement_boost = min((s.get("endorsements", 0) or 0) / 20, 0.3)
            score += weight + endorsement_boost
    return min(score / 6, 1.0)  # normalize roughly

df["skill_match_score"] = df["candidate_id"].apply(skill_match_score)

def experience_fit_score(years):
    # JD targets ~5-9 yrs senior band; soft penalty outside
    if 5 <= years <= 9:
        return 1.0
    elif 3 <= years < 5 or 9 < years <= 12:
        return 0.6
    else:
        return 0.3

df["experience_fit_score"] = df["years_of_experience"].apply(experience_fit_score)

In [34]:
def behavioral_multiplier(row):
    mult = 1.0
    # recency penalty
    if row["days_since_active"] > 180:
        mult *= 0.6
    elif row["days_since_active"] > 90:
        mult *= 0.85
    # responsiveness boost/penalty
    mult *= (0.7 + 0.5 * (row["recruiter_response_rate"] or 0))  # 0.7x to 1.2x
    # interview completion
    if row["interview_completion_rate"] < 0.5:
        mult *= 0.9
    # open to work boost
    if row["open_to_work_flag"]:
        mult *= 1.05
    return round(mult, 3)

df["behavioral_multiplier"] = df.apply(behavioral_multiplier, axis=1)

In [35]:
df["base_score"] = (
    0.35 * df["title_score"] +
    0.30 * df["skill_match_score"] +
    0.20 * df["career_substance_score"] +
    0.15 * df["experience_fit_score"]
)
df["final_score"] = df["base_score"] * df["behavioral_multiplier"]

ranked = df[~df["is_honeypot"]].sort_values("final_score", ascending=False).reset_index(drop=True)
top_100 = ranked.head(100).copy()
top_100["rank"] = range(1, 101)

print(top_100[["rank", "candidate_id", "current_title", "final_score", "honeypot_flags"]].head(15).to_string())

    rank  candidate_id                     current_title  final_score honeypot_flags
0      1  CAND_0077337   Staff Machine Learning Engineer     1.104430             []
1      2  CAND_0046525  Senior Machine Learning Engineer     1.101240             []
2      3  CAND_0055905  Senior Machine Learning Engineer     1.048960             []
3      4  CAND_0088025   Staff Machine Learning Engineer     1.030480             []
4      5  CAND_0080766   Staff Machine Learning Engineer     1.030000             []
5      6  CAND_0071974                Senior AI Engineer     1.005480             []
6      7  CAND_0024990                Junior ML Engineer     0.991447             []
7      8  CAND_0016163               Applied ML Engineer     0.979440             []
8      9  CAND_0069905               Applied ML Engineer     0.960960             []
9     10  CAND_0064256                Junior ML Engineer     0.957201             []
10    11  CAND_0094482                Junior ML Engineer     0.95

In [36]:
import os
os.makedirs("/content/drive/MyDrive/india_runs/outputs", exist_ok=True)
df.to_csv("/content/drive/MyDrive/india_runs/outputs/all_candidates_features.csv", index=False)
top_100.to_csv("/content/drive/MyDrive/india_runs/outputs/top_100_draft.csv", index=False)
print("Saved to Google Drive: outputs/all_candidates_features.csv and outputs/top_100_draft.csv")

Saved to Google Drive: outputs/all_candidates_features.csv and outputs/top_100_draft.csv


In [37]:
def generate_reasoning(row, cid):
    parts = []

    # Title/role fit
    if row["title_score"] >= 1.0:
        parts.append(f"currently {row['current_title']}, a direct match for the target role")
    elif row["title_score"] >= 0.7:
        parts.append(f"currently {row['current_title']}, an adjacent AI/ML role")
    else:
        parts.append(f"currently {row['current_title']}, transitioning toward AI/ML based on skills and projects")

    # Skill strength — pull top 2 relevant skills with longest duration
    skills = skills_lookup.get(cid, [])
    relevant_kw = ["python", "pytorch", "tensorflow", "machine learning", "deep learning",
                   "nlp", "llm", "transformers", "computer vision", "mlops", "sql", "spark"]
    rel_skills = [s for s in skills if any(k in (s.get("name") or "").lower() for k in relevant_kw)]
    rel_skills_sorted = sorted(rel_skills, key=lambda s: s.get("duration_months", 0) or 0, reverse=True)
    if rel_skills_sorted:
        top_skill_names = ", ".join(s["name"] for s in rel_skills_sorted[:2])
        parts.append(f"strongest hands-on skills include {top_skill_names}")

    # Career substance
    if row["career_substance_score"] >= 0.6:
        parts.append("career history shows clear production/deployment experience")
    elif row["career_substance_score"] >= 0.3:
        parts.append("some production-facing work evident in career history")

    # Experience fit
    parts.append(f"{row['years_of_experience']:.1f} years of experience")

    # Behavioral signal note
    if row["days_since_active"] <= 30:
        parts.append("recently active on the platform")
    elif row["days_since_active"] > 180:
        parts.append("note: inactive for an extended period")

    return "; ".join(parts) + "."

top_100["reasoning"] = top_100.apply(lambda r: generate_reasoning(r, r["candidate_id"]), axis=1)
print(top_100[["rank", "candidate_id", "reasoning"]].head(5).to_string())

   rank  candidate_id                                                                                                                                                                                                                                                                          reasoning
0     1  CAND_0077337                              currently Staff Machine Learning Engineer, a direct match for the target role; strongest hands-on skills include Python, LLMs; career history shows clear production/deployment experience; 7.0 years of experience; recently active on the platform.
1     2  CAND_0046525  currently Senior Machine Learning Engineer, a direct match for the target role; strongest hands-on skills include Sentence Transformers, Machine Learning; career history shows clear production/deployment experience; 6.1 years of experience; recently active on the platform.
2     3  CAND_0055905                                                  currently Senior Machine Learning Engi

In [38]:
# Submission CSV exactly as required
submission = top_100[["rank", "candidate_id", "final_score", "reasoning"]].copy()
submission.columns = ["rank", "candidate_id", "score", "reasoning"]

# Save to Drive
submission.to_csv("/content/drive/MyDrive/india_runs/outputs/final_submission.csv", index=False)
print(submission.head())
print(f"\nTotal rows: {len(submission)}")
print(f"Score non-increasing check: {all(submission['score'].diff().dropna() <= 0)}")

   rank  candidate_id    score  \
0     1  CAND_0077337  1.10443   
1     2  CAND_0046525  1.10124   
2     3  CAND_0055905  1.04896   
3     4  CAND_0088025  1.03048   
4     5  CAND_0080766  1.03000   

                                           reasoning  
0  currently Staff Machine Learning Engineer, a d...  
1  currently Senior Machine Learning Engineer, a ...  
2  currently Senior Machine Learning Engineer, a ...  
3  currently Staff Machine Learning Engineer, a d...  
4  currently Staff Machine Learning Engineer, a d...  

Total rows: 100
Score non-increasing check: True


In [40]:
# Copy validator from Drive to Colab local, then run
import shutil
shutil.copy(
    "/content/drive/MyDrive/india_runs/India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/validate_submission.py",
    "/content/validate_submission.py"
)

!python /content/validate_submission.py \
    --submission "/content/drive/MyDrive/india_runs/outputs/final_submission.csv" \
    --candidates "/content/drive/MyDrive/india_runs/India_runs_data_and_ai_challenge/[PUB] India_runs_data_and_ai_challenge/India_runs_data_and_ai_challenge/candidates.jsonl"

Usage: python validate_submission.py <participant_id>.csv


In [41]:
import shutil

# Step 1: CSV ko team ID ke naam se rename karke copy kar
TEAM_ID = "AbstraX"  # ya jo bhi hack2skill pe team ID dikhi ho (check karo submission page pe)

submission_filename = f"/content/{TEAM_ID}.csv"
shutil.copy("/content/drive/MyDrive/india_runs/outputs/final_submission.csv", submission_filename)

# Step 2: Validator sahi format mein chalao
!python /content/validate_submission.py "{submission_filename}"

Validation failed (202 issue(s)):

- Row 1 (header) must be exactly:
  candidate_id,rank,score,reasoning
Found:
  rank,candidate_id,score,reasoning
- Row 2: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 2: rank must be an integer (1–100).
- Row 3: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 3: rank must be an integer (1–100).
- Row 4: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 4: rank must be an integer (1–100).
- Row 5: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 5: rank must be an integer (1–100).
- Row 6: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 6: rank must be an integer (1–100).
- Row 7: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 7: rank must be an integer (1–100).
- Row 8: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 8: rank must be an integer (1–100).
- Row 9: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 9: rank must be an integer (1–100).
- Row 10: candidate_id must be CAND_XXXXXXX (7 digits).
- Row 10: rank must 

In [42]:
# Fix column order exactly as validator expects
submission = top_100[["candidate_id", "rank", "final_score", "reasoning"]].copy()
submission.columns = ["candidate_id", "rank", "score", "reasoning"]

# Verify
print(submission.columns.tolist())  # must be ['candidate_id', 'rank', 'score', 'reasoning']
print(submission.head(3))
print(f"Total rows: {len(submission)}")

# Save
submission.to_csv("/content/drive/MyDrive/india_runs/outputs/final_submission.csv", index=False)

# Copy with team name and re-validate
import shutil
shutil.copy("/content/drive/MyDrive/india_runs/outputs/final_submission.csv", "/content/AbstraX.csv")
!python /content/validate_submission.py "/content/AbstraX.csv"

['candidate_id', 'rank', 'score', 'reasoning']
   candidate_id  rank    score  \
0  CAND_0077337     1  1.10443   
1  CAND_0046525     2  1.10124   
2  CAND_0055905     3  1.04896   

                                           reasoning  
0  currently Staff Machine Learning Engineer, a d...  
1  currently Senior Machine Learning Engineer, a ...  
2  currently Senior Machine Learning Engineer, a ...  
Total rows: 100
Validation failed (1 issue(s)):

- Equal scores at ranks 96 and 97: tie-break requires candidate_id ascending ('CAND_0067535' > 'CAND_0053591').


In [43]:
# Sort by score desc, then candidate_id asc for tie-breaking
ranked_final = df[~df["is_honeypot"]].sort_values(
    ["final_score", "candidate_id"],
    ascending=[False, True]
).reset_index(drop=True)

top_100 = ranked_final.head(100).copy()
top_100["rank"] = range(1, 101)
top_100["reasoning"] = top_100.apply(lambda r: generate_reasoning(r, r["candidate_id"]), axis=1)

# Final CSV
submission = top_100[["candidate_id", "rank", "final_score", "reasoning"]].copy()
submission.columns = ["candidate_id", "rank", "score", "reasoning"]

submission.to_csv("/content/drive/MyDrive/india_runs/outputs/final_submission.csv", index=False)

import shutil
shutil.copy("/content/drive/MyDrive/india_runs/outputs/final_submission.csv", "/content/AbstraX.csv")
!python /content/validate_submission.py "/content/AbstraX.csv"

Submission is valid.
